In [1]:
import scanpy as sc
import cellxgene_census
import cellxgene_census.experimental as czi_exp
import tiledb
import os
import time

In [2]:
# Verify that the proxy settings are correctly set
http_proxy = os.getenv('http_proxy')
https_proxy = os.getenv('https_proxy')

print("HTTP_PROXY: {}".format(http_proxy))
print("HTTPS_PROXY: {}".format(https_proxy))

um_proxy = "proxy1.arc-ts.umich.edu" ##### this is the setup used below in the open_soma call
um_proxy_port = "3128"

config= {
    "vfs.s3.proxy_host": um_proxy, 
    "vfs.s3.proxy_port": um_proxy_port,
    "vfs.s3.request_timeout_ms": 1000000, 
}

HTTP_PROXY: http://proxy1.arc-ts.umich.edu:3128/
HTTPS_PROXY: http://proxy1.arc-ts.umich.edu:3128/


In [3]:
cellxgene_census.get_census_version_description('stable')

{'release_date': None,
 'release_build': '2025-01-30',
 'soma': {'uri': 's3://cellxgene-census-public-us-west-2/cell-census/2025-01-30/soma/',
  'relative_uri': '/cell-census/2025-01-30/soma/',
  's3_region': 'us-west-2'},
 'h5ads': {'uri': 's3://cellxgene-census-public-us-west-2/cell-census/2025-01-30/h5ads/',
  'relative_uri': '/cell-census/2025-01-30/h5ads/',
  's3_region': 'us-west-2'},
 'flags': {'lts': True}}

# Extract metadata

In [4]:
# organism = "mus_musculus"
organism = "homo_sapiens"

with cellxgene_census.open_soma(census_version="2024-07-01", tiledb_config=config) as census: 

    # Reads SOMADataFrame as a slice
    cell_metadata = census["census_data"][organism].obs.read()

    # Concatenates results to pyarrow.Table
    cell_metadata = cell_metadata.concat()
    
    # Converts to pandas.DataFrame
    cell_metadata = cell_metadata.to_pandas()
    census.close()

print(f"{cell_metadata.shape=}")

outpath = "/nfs/turbo/umms-indikar/shared/projects/czi_projects/metadata/all_metadata.parquet"
cell_metadata.to_parquet(outpath)
cell_metadata.head()

cell_metadata.shape=(74322510, 28)


,soma_joinid,dataset_id,assay,assay_ontology_term_id,cell_type,cell_type_ontology_term_id,development_stage,development_stage_ontology_term_id,disease,disease_ontology_term_id,...,tissue,tissue_ontology_term_id,tissue_type,tissue_general,tissue_general_ontology_term_id,raw_sum,nnz,raw_mean_nnz,raw_variance_nnz,n_measured_vars
0,0,0895c838-e550-48a3-a777-dbcd35d30272,10x 3' v2,EFO:0009899,plasma cell,CL:0000786,human adult stage,HsapDv:0000087,normal,PATO:0000461,...,caudate lobe of liver,UBERON:0001117,tissue,liver,UBERON:0002107,1742.0,221,7.882353,6661.549733,13696
1,1,0895c838-e550-48a3-a777-dbcd35d30272,10x 3' v2,EFO:0009899,mature B cell,CL:0000785,human adult stage,HsapDv:0000087,normal,PATO:0000461,...,caudate lobe of liver,UBERON:0001117,tissue,liver,UBERON:0002107,278.0,143,1.944056,3.616567,13696
2,2,0895c838-e550-48a3-a777-dbcd35d30272,10x 3' v2,EFO:0009899,plasma cell,CL:0000786,human adult stage,HsapDv:0000087,normal,PATO:0000461,...,caudate lobe of liver,UBERON:0001117,tissue,liver,UBERON:0002107,3723.0,709,5.251058,2959.510327,13696
3,3,0895c838-e550-48a3-a777-dbcd35d30272,10x 3' v2,EFO:0009899,mature B cell,CL:0000785,human adult stage,HsapDv:0000087,normal,PATO:0000461,...,caudate lobe of liver,UBERON:0001117,tissue,liver,UBERON:0002107,351.0,179,1.960894,8.487226,13696
4,4,0895c838-e550-48a3-a777-dbcd35d30272,10x 3' v2,EFO:0009899,mature B cell,CL:0000785,human adult stage,HsapDv:0000087,normal,PATO:0000461,...,caudate lobe of liver,UBERON:0001117,tissue,liver,UBERON:0002107,605.0,275,2.200000,18.978102,13696


In [5]:
df = cell_metadata[(cell_metadata['is_primary_data']) & (cell_metadata['disease'] == 'normal')].copy()
print(f"{df.shape=}")

column = 'cell_type'
print(df[column].nunique())
df[column].value_counts()

df.shape=(30197419, 28)
664


cell_type
neuron                                                               2692914
glutamatergic neuron                                                 1566247
L2/3-6 intratelencephalic projecting glutamatergic neuron            1104521
oligodendrocyte                                                       901217
fibroblast                                                            715861
                                                                      ...   
nephron tubule epithelial cell                                             0
oocyte                                                                     0
pyramidal neuron                                                           0
smooth muscle cell of large intestine                                      0
CD56-positive, CD161-positive immature natural killer cell, human          0
Name: count, Length: 698, dtype: int64

# Save the metadata

In [6]:
outpath = f"/nfs/turbo/umms-indikar/shared/projects/czi_projects/metadata/{organism}.parquet"
df.to_parquet(outpath)
df.head()

,soma_joinid,dataset_id,assay,assay_ontology_term_id,cell_type,cell_type_ontology_term_id,development_stage,development_stage_ontology_term_id,disease,disease_ontology_term_id,...,tissue,tissue_ontology_term_id,tissue_type,tissue_general,tissue_general_ontology_term_id,raw_sum,nnz,raw_mean_nnz,raw_variance_nnz,n_measured_vars
4308,4308,a5d95a42-0137-496f-8a60-101e17f263c8,Smart-seq2,EFO:0008931,naive B cell,CL:0000788,26-year-old human stage,HsapDv:0000120,normal,PATO:0000461,...,blood,UBERON:0000178,tissue,blood,UBERON:0000178,17691.0,1199,14.754796,624.620960,19149
4309,4309,a5d95a42-0137-496f-8a60-101e17f263c8,Smart-seq2,EFO:0008931,naive B cell,CL:0000788,26-year-old human stage,HsapDv:0000120,normal,PATO:0000461,...,blood,UBERON:0000178,tissue,blood,UBERON:0000178,13612.0,841,16.185493,1394.944123,19149
4310,4310,a5d95a42-0137-496f-8a60-101e17f263c8,Smart-seq2,EFO:0008931,naive B cell,CL:0000788,26-year-old human stage,HsapDv:0000120,normal,PATO:0000461,...,blood,UBERON:0000178,tissue,blood,UBERON:0000178,126774.0,1683,75.326203,42215.146798,19149
4311,4311,a5d95a42-0137-496f-8a60-101e17f263c8,Smart-seq2,EFO:0008931,naive B cell,CL:0000788,26-year-old human stage,HsapDv:0000120,normal,PATO:0000461,...,blood,UBERON:0000178,tissue,blood,UBERON:0000178,14370.0,371,38.733154,4639.990763,19149
4312,4312,a5d95a42-0137-496f-8a60-101e17f263c8,Smart-seq2,EFO:0008931,naive B cell,CL:0000788,26-year-old human stage,HsapDv:0000120,normal,PATO:0000461,...,blood,UBERON:0000178,tissue,blood,UBERON:0000178,13738.0,507,27.096647,3874.079574,19149


In [7]:
df.columns

Index(['soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id',
       'cell_type', 'cell_type_ontology_term_id', 'development_stage',
       'development_stage_ontology_term_id', 'disease',
       'disease_ontology_term_id', 'donor_id', 'is_primary_data',
       'observation_joinid', 'self_reported_ethnicity',
       'self_reported_ethnicity_ontology_term_id', 'sex',
       'sex_ontology_term_id', 'suspension_type', 'tissue',
       'tissue_ontology_term_id', 'tissue_type', 'tissue_general',
       'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz',
       'raw_variance_nnz', 'n_measured_vars'],
      dtype='object')

In [8]:
for column in df.columns:
    print(f"---- {column} ----")
    print(f"unique values {df[column].nunique()}")
    

---- soma_joinid ----
unique values 30197419
---- dataset_id ----
unique values 305
---- assay ----
unique values 21
---- assay_ontology_term_id ----
unique values 21
---- cell_type ----
unique values 664
---- cell_type_ontology_term_id ----
unique values 664
---- development_stage ----
unique values 172
---- development_stage_ontology_term_id ----
unique values 172
---- disease ----
unique values 1
---- disease_ontology_term_id ----
unique values 1
---- donor_id ----
unique values 3790
---- is_primary_data ----
unique values 1
---- observation_joinid ----
unique values 30122637
---- self_reported_ethnicity ----
unique values 28
---- self_reported_ethnicity_ontology_term_id ----
unique values 28
---- sex ----
unique values 3
---- sex_ontology_term_id ----
unique values 3
---- suspension_type ----
unique values 2
---- tissue ----
unique values 233
---- tissue_ontology_term_id ----
unique values 233
---- tissue_type ----
unique values 1
---- tissue_general ----
unique values 52
---- tiss